## Stage 1 — 트랙 수집 (YOLO + BotSort)

비디오에서 YOLO로 사람을 탐지하고 BotSort로 track_id를 부여합니다.

출력: `tracks/{영상이름}.pkl`

```python
{
  'tracks': {track_id: [{'frame': int, 'bbox': [x1,y1,x2,y2], 'conf': float}]},
  'frames_processed': int,
  'video_path': str,
  'git_commit': str,
  'git_branch': str,
}
```

- 1000프레임마다 체크포인트 자동 저장
- 재실행 시 중단된 프레임부터 자동 재개

In [ ]:
import sys as _sys, os as _os

IN_COLAB = 'google.colab' in _sys.modules or _os.environ.get('EYE_D_COLAB') == '1'

if IN_COLAB:
    if 'google.colab' in _sys.modules:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=True)
        import subprocess
        subprocess.run(['pip', 'install', '-q', 'ultralytics', 'boxmot'], check=True)


In [ ]:
import os, sys, cv2, numpy as np
from collections import defaultdict

IN_COLAB = globals().get('IN_COLAB', 'google.colab' in sys.modules or os.environ.get('EYE_D_COLAB') == '1')

if IN_COLAB:
    _drive_root = globals().get('DRIVE_PROJECT_ROOT', os.environ.get('EYE_D_DRIVE_ROOT',
        '/content/drive/MyDrive/projects/EYE-D/EYE-D'))
    edge_root = f'{_drive_root}/edge'
else:
    current_dir = os.path.abspath(os.getcwd())
    edge_root = os.path.abspath(os.path.join(current_dir, '..')) if os.path.basename(current_dir) == 'notebooks' else os.path.join(current_dir, 'edge')

if edge_root not in sys.path:
    sys.path.insert(0, edge_root)

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}  |  edge_root: {edge_root}')


In [ ]:
import pathlib as _pl

# ── papermill -p 로 주입되는 파라미터 ────────────────────────────────────────
GIT_COMMIT  = "unknown"
GIT_BRANCH  = "unknown"

if IN_COLAB:
    VIDEO_PATH = os.path.join(globals().get('VIDEO_DIR', '/content/drive/MyDrive/EYE-D/data'), '14300002.avi')
    _tracks_dir = globals().get('TRACKS_DIR', '/content/drive/MyDrive/EYE-D/tracks')
else:
    VIDEO_PATH  = os.path.abspath(os.path.join(edge_root, '..', 'data', '14300002.avi'))
    _tracks_dir = os.path.abspath(os.path.join(edge_root, '..', 'tracks'))

MAX_FRAMES          = float('inf')
CONF_THRESH         = 0.40
MIN_BBOX_SIZE       = 40
TRACKS_DIR          = _tracks_dir
CHECKPOINT_INTERVAL = 1000
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
import subprocess as _sp

if GIT_COMMIT == "unknown":
    try:
        GIT_COMMIT = _sp.check_output(['git', 'rev-parse', 'HEAD'], cwd=edge_root, stderr=_sp.DEVNULL).decode().strip()
        GIT_BRANCH = _sp.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], cwd=edge_root, stderr=_sp.DEVNULL).decode().strip()
    except Exception:
        GIT_COMMIT, GIT_BRANCH = 'unknown', 'unknown'

print(f'git branch : {GIT_BRANCH}')
print(f'git commit : {GIT_COMMIT[:12]}...')


In [ ]:
TRACKS_PKL = str(_pl.Path(TRACKS_DIR) / f'{_pl.Path(VIDEO_PATH).stem}.pkl')

if not os.path.exists(VIDEO_PATH):
    raise FileNotFoundError(f'비디오 없음: {VIDEO_PATH}')

_cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(_cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps          = _cap.get(cv2.CAP_PROP_FPS)
_cap.release()

_analyze = total_frames if MAX_FRAMES == float('inf') else int(min(MAX_FRAMES, total_frames))
print(f'비디오  : {os.path.basename(VIDEO_PATH)}')
print(f'전체    : {total_frames}프레임  |  {fps:.1f} FPS  |  {total_frames/fps:.0f}초')
print(f'처리 예정: {_analyze}프레임')
print(f'tracks  : {TRACKS_PKL}')

# 재개 여부 확인
_resume_from    = 0
_already_done   = False
if os.path.exists(TRACKS_PKL):
    import pickle as _pk
    with open(TRACKS_PKL, 'rb') as _f:
        _ckpt = _pk.load(_f)
    _frames_done = _ckpt.get('frames_processed', 0)
    if _frames_done >= _analyze:
        print(f'\n[완료] 이미 처리됨 ({_frames_done}프레임) — 수집 단계를 건너뜁니다.')
        _already_done = True
    else:
        _resume_from = _frames_done
        print(f'\n[재개] {_frames_done} / {_analyze} 프레임 처리됨 → 이어서 실행')


In [ ]:
try:
    from ultralytics import YOLO
    from boxmot.trackers.tracker_zoo import create_tracker
    TRACKER_AVAILABLE = True
    print('추론 라이브러리 로드 완료')
except ImportError as e:
    raise RuntimeError(f'Stage 1 실행에 ultralytics, boxmot 필요:\n  {e}')


def collect_tracks(video_path, max_frames=float('inf'), verbose=True,
                   start_frame=0, checkpoint_path=None, checkpoint_interval=1000,
                   existing_tracks=None):
    """
    YOLO + BotSort로 track_id와 bbox를 수집합니다. ReID 없음.

    Returns
    -------
    (dict[int, list[{'frame', 'bbox', 'conf'}]], int)
    """
    import pickle as _ckpt_pickle, pathlib as _ckpt_pl

    half = (DEVICE != 'cpu')
    detector = YOLO('yolov8n.pt')
    tracker  = create_tracker('botsort', reid_weights='osnet_x0_25_msmt17.pt',
                               device=DEVICE, half=half)

    cap = cv2.VideoCapture(video_path)
    if start_frame > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
        if verbose:
            print(f'  프레임 {start_frame}부터 재개')

    track_data = defaultdict(list)
    if existing_tracks:
        for tid, recs in existing_tracks.items():
            track_data[tid].extend(recs)

    frame_idx  = start_frame
    _last_ckpt = start_frame

    while cap.isOpened() and frame_idx < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1

        det_results = detector.predict(frame, conf=CONF_THRESH, classes=[0], verbose=False)
        if len(det_results[0].boxes) == 0:
            tracker.update(np.empty((0, 6)), frame)
            continue

        dets   = det_results[0].boxes.data.cpu().numpy()
        tracks = tracker.update(dets, frame)
        if len(tracks) == 0:
            continue

        for t in tracks:
            if t[5] < CONF_THRESH:
                continue
            bbox = [int(t[0]), int(t[1]), int(t[2]), int(t[3])]
            w, h = bbox[2]-bbox[0], bbox[3]-bbox[1]
            if w < MIN_BBOX_SIZE or h < MIN_BBOX_SIZE:
                continue
            track_data[int(t[4])].append({
                'frame': frame_idx,
                'bbox':  bbox,
                'conf':  float(t[5]),
            })

        if checkpoint_path and (frame_idx - _last_ckpt) >= checkpoint_interval:
            _ckpt_pl.Path(checkpoint_path).parent.mkdir(parents=True, exist_ok=True)
            with open(checkpoint_path, 'wb') as _cf:
                _ckpt_pickle.dump({
                    'tracks':           dict(track_data),
                    'frames_processed': frame_idx,
                    'video_path':       video_path,
                    'git_commit':       GIT_COMMIT,
                    'git_branch':       GIT_BRANCH,
                }, _cf)
            _last_ckpt = frame_idx
            if verbose:
                print(f'  [체크포인트] {frame_idx} 프레임 → {checkpoint_path}')

    cap.release()
    n = sum(len(v) for v in track_data.values())
    if verbose:
        print(f'  완료 → 트랙 {len(track_data)}개  |  탐지 {n}개')
    return dict(track_data), frame_idx


print('collect_tracks 함수 정의 완료')


In [ ]:
import pickle as _pickle

if _already_done:
    print('[건너뜀] 이미 완료된 tracks.pkl 존재')
else:
    _existing = None
    if _resume_from > 0 and os.path.exists(TRACKS_PKL):
        with open(TRACKS_PKL, 'rb') as _f:
            _existing = _pickle.load(_f).get('tracks')

    _pl.Path(TRACKS_DIR).mkdir(parents=True, exist_ok=True)

    tracks, frames_processed = collect_tracks(
        VIDEO_PATH,
        max_frames=MAX_FRAMES,
        start_frame=_resume_from,
        checkpoint_path=TRACKS_PKL,
        checkpoint_interval=CHECKPOINT_INTERVAL,
        existing_tracks=_existing,
    )


In [ ]:
import pickle, pathlib

if _already_done:
    print('[건너뜀] 저장 단계 스킵')
else:
    out_path = pathlib.Path(TRACKS_PKL)
    with open(out_path, 'wb') as _f:
        pickle.dump({
            'tracks':           tracks,
            'frames_processed': frames_processed,
            'total_frames':     total_frames,
            'video_path':       VIDEO_PATH,
            'git_commit':       GIT_COMMIT,
            'git_branch':       GIT_BRANCH,
        }, _f)

    n_dets = sum(len(v) for v in tracks.values())
    print(f'저장 완료: {out_path}')
    print(f'  트랙 수    : {len(tracks)}')
    print(f'  총 탐지    : {n_dets}')
    print(f'  처리 프레임: {frames_processed} / {total_frames}')
    print(f'  git commit : {GIT_COMMIT}')
